In [ ]:
# 트랜스포머에 대한 구글 AI 블로그 포스트에서 가져온 예시 문장
import tensorflow as tf
import numpy as np
import math

np.set_printoptions(precision=4, suppress=True)

# 입력 문장
tokens = [
    "The",
    "animal",
    "didn't",
    "cross",
    "the",
    "street",
    "because",
    "it",
    "was",
    "too",
    "tired",
    "."
]

vocab = list(dict.fromkeys(tokens))
print(vocab)
word_to_id = {word:i for i, word in enumerate(vocab)}
print(word_to_id)
id_to_word = {i:word for word, i in word_to_id.items()}
print(id_to_word)

input_ids = tf.constant(   # 문장을 정수 인덱스 텐서로 변환
    [word_to_id[word] for word in tokens], # 각 단어를 숫자로 변환
    dtype=tf.int32
)

it_pos = tokens.index("it")
animal_pos = tokens.index("animal")
print("it 위치 : ", it_pos)
print("animal 위치 : ", animal_pos)


In [ ]:
# Self-Attention 모델
class MySelfAttention(tf.keras.Model):
    def __init__(self, vocab_size, embed_dim):  # 매개변수 : 단어 수와 임베딩차원
        super().__init__()

        # 단어 번호를 단어 벡터로 바꾸는 임베딩 레이어
        self.embedding = tf.keras.layers.Embedding(
            input_dim = vocab_size,
            output_dim = embed_dim
        )

        # 임베딩 벡터를 Query로 바꾸는 Dense layer
        self.Wq = tf.keras.layers.Dense(
            embed_dim,
            use_bias = False
        )

        self.Wk = tf.keras.layers.Dense(
            embed_dim,
            use_bias = False
        )

        self.Wv = tf.keras.layers.Dense(
            embed_dim,
            use_bias = False
        )

    def call(self, input_ids, query_pos=None):  # 모델이 호출될 때 자동 실행
        # 단어 인덱스를 임베딩벡터로 변환
        x = self.embedding(input_ids)

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # attention score
        scores = tf.matmul(
            Q, K, transpose_b=True     # K를 전치하여 계산
        ) / math.sqrt(K.shape[-1])     # score가 너무 커지지 않도록 루트 dk로 나눔

        # "it" 위치의 Query가 모든 단어 Key를 본 점수 추출
        query_scores = scores[query_pos]
        self_mask = tf.one_hot(
            query_pos,   # 마스크할 위치(it 위치)
            depth=tf.shape(input_ids)[0]   # 전체 토큰 갯수
        ) * -1e9    # 해당 위치 점수를 매우 작은 값으로 만들어 softmax 결과가 거의 0이 되게 함

        query_scores = query_scores + self_mask  # "it" 자기 자신 위치의 score를 작게 만듦
        attention_weights = tf.nn.softmax(query_scores)

        # attention_weight 만큼 Value 가중합 구하기
        context_vector = tf.reduce_sum(attention_weights[:, tf.newaxis] * V, axis=0)

        return query_scores, attention_weights, context_vector, x, Q, K, V



In [ ]:
# 모델 생성 및 학습 전 상태 확인
vocab_size = len(vocab)
embed_dim = 8

model = MySelfAttention(vocab_size=vocab_size, embed_dim=embed_dim)

# query_scores, attention_weights, context_vector, x, Q, K, V
_, weights_before, _, embeddings_before, _, _, _ = model(input_ids, query_pos=it_pos)

Wq_before = model.Wq.get_weights()[0].copy()  # 학습 전 Wq 가중치 복사
# print('Wq_before : ', Wq_before)
Wk_before = model.Wk.get_weights()[0].copy()  # 학습 전 Wk 가중치 복사
Wv_before = model.Wv.get_weights()[0].copy()  # 학습 전 Wv 가중치 복사

print('[학습전 임베딩 일부]')
print('animal embedding : ', embeddings_before[animal_pos].numpy())
print('it embedding : ', embeddings_before[it_pos].numpy())

print('[학습전 it의 attention]')
for_print_before = weights_before.numpy()
for token, w in zip(tokens, for_print_before): # 각 토큰과 weight를 순회
    print(f'{token:>8} -> attention weight:{w:4f}')


In [ ]:
# 학습
optimizer = tf.keras.optimizers.Adam(learning_rate=0.05)
target = tf.constant([animal_pos], dtype=tf.int32)   # 정답 클래스는 animal 위치

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for epoch in range(500):
    with tf.GradientTape() as tape:    # 자동미분 기록
        query_scores, attention_weights, context_vector, x, Q, K, V = model(
            input_ids, query_pos=it_pos
        )
        loss = loss_fn(target, query_scores[tf.newaxis, :])  # batch 차원 추가: shape(1, seq_len)으로 만듦

    gradients = tape.gradient( # loss를 기준으로 학습 가능한 변수들의 기울기 계산
        loss,   # 줄이고 싶은 손실값
        model.trainable_variables  # embedding, Wq, Wk, Wv
    )

    optimizer.apply_gradients(  # 계산된 기울기를 이용해 가중치 갱신
        zip(gradients, model.trainable_variables)
    )

    if epoch % 100 == 0:
        print(f'epoch {epoch:3d} | loss:{loss.numpy():.5f}')



In [ ]:
# 학습 후 결과
query_scores, attention_weights, context_vector, embeddings_after, Q, K, V = model(
    input_ids, query_pos=it_pos
)

print('[학습후 임베딩 일부]')
print('animal embedding : ', embeddings_after[animal_pos].numpy())
print('it embedding : ', embeddings_after[it_pos].numpy())

print('[학습후 it의 attention]')
for token, score, w in zip(tokens, query_scores.numpy(), attention_weights.numpy()):
    print(f'{token:>8} -> score:{score:9.4f}, attention weight:{w:4f}')

print()
max_index = np.argmax(attention_weights.numpy())
print("[결론]")
print(f"'it'이 가장 많이 참고한 단어는 '{tokens[max_index]}'")
print('it의 context vector ', context_vector.numpy())